# Taller de Tesis I — Entrega 1
## Dataset, Pregunta de Investigación y Objetivo

**Maestría en Data Mining — FCEyN, UBA**  
**Grupo:** G2  
**Alumno:** [Tu nombre]  
**Dataset:** Stack Overflow Developer Survey 2023 + 2024

## 0. Configuración e imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
sns.set_theme(style='darkgrid', palette='muted')

print('✅ Imports OK')

## 1. Carga de datos

Descargá los CSVs desde https://survey.stackoverflow.co/ y colocálos en `../data/`

In [ ]:
df23 = pd.read_csv('../data/survey_results_public_2023.csv', low_memory=False)
df24 = pd.read_csv('../data/survey_results_public_2024.csv', low_memory=False)

df23['survey_year'] = 2023
df24['survey_year'] = 2024

print(f'2023: {df23.shape[0]:,} filas, {df23.shape[1]} columnas')
print(f'2024: {df24.shape[0]:,} filas, {df24.shape[1]} columnas')

## 2. Detección de columnas de IA

In [ ]:
ai_cols_23 = [c for c in df23.columns if 'AI' in c]
ai_cols_24 = [c for c in df24.columns if 'AI' in c]

print('=== Columnas de IA en 2023 ===')
print(ai_cols_23)
print('\n=== Columnas de IA en 2024 ===')
print(ai_cols_24)

## 3. Exploración de valores en columnas clave

⚠️ **Importante:** revisá los valores exactos antes de construir el target. Los labels pueden variar entre 2023 y 2024.

In [ ]:
for col in ['AISelect', 'AIAcc', 'AISent']:
    for year, df in [(2023, df23), (2024, df24)]:
        if col in df.columns:
            print(f'\n--- {col} ({year}) ---')
            print(df[col].value_counts(dropna=False))

## 4. Alineación y unificación de datasets

In [ ]:
cols_perfil = [
    'Country', 'DevType', 'YearsCodePro', 'OrgSize',
    'EdLevel', 'Employment', 'LanguageHaveWorkedWith',
    'CompTotal', 'WorkExp', 'survey_year'
]
cols_ia = ['AISelect', 'AIAcc', 'AISent', 'AIBen']

def filtrar_cols(df, cols):
    return [c for c in cols if c in df.columns]

df23_sub = df23[filtrar_cols(df23, cols_perfil + cols_ia)].copy()
df24_sub = df24[filtrar_cols(df24, cols_perfil + cols_ia)].copy()

df = pd.concat([df23_sub, df24_sub], ignore_index=True)
print(f'Dataset unificado: {df.shape[0]:,} filas, {df.shape[1]} columnas')
df.head(3)

## 5. Construcción de la variable target

**Criterio:**
- `target = 1` → usa IA actualmente en su trabajo **Y** tiene actitud favorable o muy favorable
- `target = 0` → cualquier otro caso

**Justificación:** captura "adoptante sostenido", no solo quien lo probó. Basado en el Technology Acceptance Model (TAM).

⚠️ **Ajustá los valores de las listas según lo que viste en la celda 3.**

In [ ]:
# Ajustar según valores reales observados en celda 3
usa_ia = [
    'Yes, I use AI tools in my development work currently',
    'Yes, I use AI tools currently',
]

actitud_positiva = [
    'Favorable',
    'Very favorable',
    'Highly favorable',
]

df['target'] = (
    df['AISelect'].isin(usa_ia) &
    df['AIAcc'].isin(actitud_positiva)
).astype(int)

total = len(df)
adoptantes = df['target'].sum()
print(f'Target = 1 (adoptantes):     {adoptantes:,} ({adoptantes/total:.1%})')
print(f'Target = 0 (no adoptantes):  {total-adoptantes:,} ({(total-adoptantes)/total:.1%})')

## 6. Descripción general del dataset

In [ ]:
print('=== Tipos de datos ===')
print(df.dtypes)
print(f'\n=== Valores faltantes (%) ===')
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0].round(1))

## 7. Visualizaciones para la Entrega 1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Stack Overflow Survey 2023+2024 — Exploración inicial', fontsize=13)

# 7a. Distribución del target
ax = axes[0]
counts = df['target'].value_counts()
ax.bar(['No adoptante (0)', 'Adoptante (1)'], counts.values, color=['#ef4444', '#22c55e'])
ax.set_title('Distribución del target')
ax.set_ylabel('Cantidad de respuestas')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=10)

# 7b. Tasa de adopción por año
ax = axes[1]
tasa = df.groupby('survey_year')['target'].mean() * 100
ax.bar(tasa.index.astype(str), tasa.values, color=['#3b82f6', '#8b5cf6'])
ax.set_title('Tasa de adopción por año')
ax.set_ylabel('% adoptantes')
ax.set_ylim(0, 100)
for i, v in enumerate(tasa.values):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10)

# 7c. Top 10 países
ax = axes[2]
df['Country'].value_counts().head(10).plot(kind='barh', ax=ax, color='#f59e0b')
ax.set_title('Top 10 países (respuestas)')
ax.set_xlabel('Cantidad')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../outputs/entrega1_exploracion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado en outputs/entrega1_exploracion.png')

## 8. Resumen final para el documento

In [ ]:
print('╔══════════════════════════════════════════════════════════════╗')
print('║           RESUMEN PARA DOCUMENTO DE ENTREGA 1               ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║ Pregunta:                                                    ║')
print('║   ¿Qué características del perfil de un desarrollador        ║')
print('║   predicen su adopción sostenida de herramientas de IA?      ║')
print('║                                                              ║')
print('║ Dataset: Stack Overflow Developer Survey 2023 + 2024         ║')
print('║ Fuente:  https://survey.stackoverflow.co/                    ║')
print('║ Licencia: Open Database License (ODbL)                       ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║ Filas totales:    {len(df):>10,}                               ║')
print(f'║ Adoptantes (1):   {df["target"].sum():>10,} ({df["target"].mean():.1%})                    ║')
print(f'║ No adoptantes (0):{(len(df)-df["target"].sum()):>10,} ({1-df["target"].mean():.1%})                    ║')
print('╚══════════════════════════════════════════════════════════════╝')